# Assignment 1: Classification
# Purchase Dataset

## 184.702 Machine Learning (VU 3,0) 2021W

### Group 11

**Members:** <br> András Bonifác Kónya (Student ID: 01502933) <br> Branimir Raguž (Student ID: 12123474) <br> Thummanoon Kunanuntakij (Student ID: 12122522)

## 1. Setup

In [8]:
# Imports
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn import metrics
from sklearn.metrics import f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.naive_bayes import GaussianNB
from sklearn import svm
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, GridSearchCV, RandomizedSearchCV, cross_validate, train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, Normalizer, MaxAbsScaler, OneHotEncoder
from sklearn.feature_selection import VarianceThreshold
from matplotlib import pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.decomposition import PCA
from sklearn.decomposition import TruncatedSVD

## 2. Data Loading

In [9]:
# Importing the data sets from the csv files
df_train = pd.read_csv('/kaggle/input/purchase/purchase600-100cls-15k.lrn.csv')
df_test = pd.read_csv('/kaggle/input/purchase/purchase600-100cls-15k.tes.csv')

## 3. Overview of the Data

### Description 

The purchase dataset contains information about 10000 customers and 600 binary attributes (product types), showing if a customer purchased the corresponding product or not. There are 100 classes in the target attribute. Each class represents a group of customers with similar purchase behaviour. The classification task is to predict the purchase behaviour of customers.

### Data exploration

In [ ]:
# Training set
df_train

In [ ]:
# Test set
df_test

In [ ]:
# Identifying missing values
df_train.isna().any().any()

In [ ]:
df_test.isna().any().any()

In [ ]:
# Correlation between variables

df_train.corr()

## 4. Train / Test Data

At this step, to simulate real life situation that we want to measure the performance of our model against unseen data. We split the data into modelling and testing set. We reserve the testing set to the very last step after we have chosen a model and its optimal parameters already.

We also split the modelling data into training and validation set again so we can use this for parameter tuning / model selection.

In [ ]:
# Train/test split for the submission.

# Separate the columns into dependent and independent variables

# X_train = df_train.iloc[:,:-1]
# y_train = df_train["class"]

# Submitting the solutions

# tmp_df = pd.DataFrame({
#    'ID': df_test['ID'],
#    'class': pipe.predict(df_test),
#})

#tmp_df.to_csv('predict_test.csv', index=False)

In [10]:
# Split for the model building/testing

X = df_train.iloc[:,:-1]
y = df_train.iloc[:, -1]

X_model, X_test, y_model, y_test = train_test_split(X, y, test_size=0.15, random_state=1)
X_train, X_val, y_train, y_val = train_test_split(X_model, y_model, test_size=0.2, random_state=1)


# X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.33, random_state=1) 

## 5. Data Preprocessing:

    1.) We did not need to remove any NA values, as there were none
    2.) There are no outliers as well as all the variables are either 0/1
    3.) Scaling/normalization is not required
    4.) None of the variables need encoding
    5.) The train/test split is supplied as well
    6.) The description of the data set is provided on kaggle, and there are no more  information I can get.

We will use a dimensionality reduction method considering that the data set is pretty big. <br> 
It will be done later in the model building part, because it is all combined in a pipeline for good practice.

In [11]:
# Plotting the variance captured by different n_components values in pca

pca = pca = PCA(6)
pca.fit(df_train.iloc[:,:-1])

plt.plot(np.cumsum(pca.explained_variance_ratio_))
plt.xlabel('number of components')
plt.ylabel('cumulative explained variance');

plt.savefig('pca_variance.png')

Even tough the 6 components in pca capture almost all the variance, we got far better results with 10 dimensions, so we used that as the value through the model developnment. 

## 6. Evaluation Metrics

### Accuracy

We would use mainly `Accuracy`. Because it is much easier to understand than other metric for multiclass classification.

**DELETE LATER**

Choose suitable multiple performance measures

    1.) Evaluate and analyse the performance (effectivness, efficiency)
    2.) Mark each step, and elaborate how to improve the results ?
    3.) Can I indentify any patterns/trends ?
    4.) Which method worked well and which did not and why ?
    5.) How do the results change when preprocessing strategies change? 
    6.) How sensitive is an algorithm to parameter settings?
    7.) Pay attention to your splits and settings Are there differences? Why? In which metrics? What could have caused it?

## 7. Model #1 - Random forest

### 7.1 Base Model

In [ ]:
clf=RandomForestClassifier(random_state=1)

score = cross_validate(clf, X_train, y_train, cv=5)
print('Mean accuracy of Random Forest(default)) cv:', np.mean(score['test_score']))

We will now try two different dimensionality reduction techniques. <br>
The methods that we are going to test are: 
* Principal Component Analysis
* Singular Value Decomposition

We are going to test different n_parameters.

### 7.2 Dimensionality reduction

#### 7.2.1 Principal Component Analysis

In [ ]:
pca__n_components = [4, 6, 10, 15, 25, 50, 100]

for i in pca__n_components:
    pipe = Pipeline([
    ('pca', PCA(n_components=i)),
    ('selector', VarianceThreshold()),
    ('classifier', RandomForestClassifier(random_state=1))])
    
    score = cross_validate(pipe, X_train, y_train, cv=5)
    
    print('n_components: ' + str(i) + ' | Test set score: ' + str(np.mean(score['test_score'])))

#### 7.2.2 Singular Value Decomposition

In [ ]:
svd__n_components = [4, 7, 10, 15, 25, 50, 100]

for i in svd__n_components:
    pipe = Pipeline([
    ('svd', TruncatedSVD(n_components=i)),
    ('selector', VarianceThreshold()),
    ('classifier', RandomForestClassifier(random_state=1))])
    
    score = cross_validate(pipe, X_train, y_train, cv=5)
    
    print('n_components: ' + str(i) + ' | Test set score: ' + str(np.mean(score['test_score'])))

The best dimension reduction method was the Principal Components Analysis. <br>
The best n_components value was 10 with the test score of 0.6524242424242425.

### 7.3 Parameter tuning

#### 7.3.1 n_estimators

In [ ]:
n_estimators = [10, 50, 100, 125, 150, 200, 300, 500]

for i in n_estimators:
    pipe = Pipeline([
    ('pca', PCA(n_components=10)),
    ('selector', VarianceThreshold()),
    ('classifier', RandomForestClassifier(n_estimators = i, random_state=1))])
    
    score = cross_validate(pipe, X_train, y_train, cv=5)
    
    print('n_estimators: ' + str(i) + ' | Test set score: ' + str(np.mean(score['test_score'])))

The best n_estimator value was 300 with the score of 0.668484848484848.

#### 7.3.2 criterion

In [ ]:
# Testing a different function to measure the quality of the split

pipe = Pipeline([
    ('pca', PCA(n_components=10)),
    ('selector', VarianceThreshold()),
    ('classifier', RandomForestClassifier(n_estimators = 300, criterion='entropy', random_state=1))])
    
    score = cross_validate(pipe, X_train, y_train, cv=5)
    
    print('Test set score: ' + str(np.mean(score['test_score'])))

We are staying with the default criterion value of 'gini'.

#### 7.3.3 max_depth

In [ ]:
max_depth = [10, 50, 75, 100]

for i in max_depth:
    pipe = Pipeline([
    ('pca', PCA(n_components=10)),
    ('selector', VarianceThreshold()),
    ('classifier', RandomForestClassifier(n_estimators = 300, random_state=1))])
    
    score = cross_validate(pipe, X_train, y_train, cv=5)
    
    print('max_depth: ' + str(i) + ' | Test set score: ' + str(np.mean(score['test_score'])))

We will stay with the max_depth default value which is none, because neither one of these values provided better results. <br>
It's best to go with the the value none because that way the nodes are expanded until all leaves are pure or until all leaves contain less than min_samples_split samples.

#### 7.3.4 min_samples_split

In [ ]:
min_samples_split = [2, 5, 10]

for i in min_samples_split:
    pipe = Pipeline([
    ('pca', PCA(n_components=10)),
    ('selector', VarianceThreshold()),
    ('classifier', RandomForestClassifier(n_estimators = 300, min_samples_split=i, random_state=1))])
    
    score = cross_validate(pipe, X_train, y_train, cv=5)
    
    print('min_samples_split: ' + str(i) + ' | Test set score: ' + str(np.mean(score['test_score'])))

Again the default value is probably the best option to go with.

#### 7.3.5 max_features

In [ ]:
max_features = ['auto', 'sqrt', 'log2']

for i in max_features:
    pipe = Pipeline([
    ('pca', PCA(n_components=10)),
    ('selector', VarianceThreshold()),
    ('classifier', RandomForestClassifier(n_estimators = 300, max_features=i, random_state=1))])
    
    score = cross_validate(pipe, X_train, y_train, cv=5)
    
    print('max_features: ' + str(i) + ' | Test set score: ' + str(np.mean(score['test_score'])))

Again, the best value seems to be the default one.
<br>
The optimal model in the end is with the n_estimators value of 300, and the pca components value of 10.

### 7.4 Optimal model

In [14]:
optimal_random_forest_classifier = Pipeline([
    ('pca', PCA(n_components=10)),
    ('selector', VarianceThreshold()),
    ('classifier', RandomForestClassifier(random_state=1, n_estimators=300, n_jobs=-1))])
    
optimal_random_forest_classifier.fit(X_train, y_train)
optimal_random_forest_predict = optimal_random_forest_classifier.predict(X_val)
    
print('Test set score: ' + str(np.mean(score['test_score'])))

## 8. Model #2 - K-Nearest Neighbor

### 8.1 Base Model

In [13]:
pipe = Pipeline([
    ('pca', PCA(n_components=10)),
    ('selector', VarianceThreshold()),
    ('classifier', KNeighborsClassifier(n_neighbors=9))
])

score = cross_validate(pipe, X_train, y_train, cv=5)
    
print('Test set score: ' + str(np.mean(score['test_score'])))

The base model gives out awful results, clearly the default settings don't work well in this case.

### 8.2 Parameter tuning

#### 8.2.1 metrics

In [ ]:
metrics = ['correlation', 'chebyshev', 'minkowski']

for i in metrics:
    pipe = Pipeline([
    ('pca', PCA(n_components=10)),
    ('selector', VarianceThreshold()),
    ('classifier', KNeighborsClassifier(n_neighbors=9, metric=i))])
    
    score = cross_validate(pipe, X_train, y_train, cv=5)
    
    print('metrics: ' + str(i) + ' | Test set score: ' + str(np.mean(score['test_score'])))

A lot of different methics wouldn't even work with this data set, but the correlation is cleary the winner.

#### 8.2.2 n_neighbors

In [ ]:
n_neighbors = np.arange(1, 21, 2)

for i in n_neighbors:
    pipe = Pipeline([
    ('pca', PCA(n_components=10)),
    ('selector', VarianceThreshold()),
    ('classifier', KNeighborsClassifier(n_neighbors=i,  metric="correlation"))])
    
    score = cross_validate(pipe, X_train, y_train, cv=5)
    
    print('n_neighbors: ' + str(i) + ' | Test set score: ' + str(np.mean(score['test_score'])))

The best score was 0.36818181818181817 with the k_neighbors value of 19. <br>
The best KNN model in the end is with n_neighbors value of 19 and the correlation used as the metric.

### 8.3 Optimal model

In [15]:
pipe = Pipeline([
    ('pca', PCA(n_components=10)),
    ('selector', VarianceThreshold()),
    ('classifier', KNeighborsClassifier(n_neighbors=19, metric="correlation"))
])

score = cross_validate(pipe, X_train, y_train, cv=5)
    
print('Test set score: ' + str(np.mean(score['test_score'])))

## 9. Model #3 - Logistic Regression

### 9.1 Base Model

In [ ]:
pipe = Pipeline([
    ('pca', PCA(n_components=10)),
    ('selector', VarianceThreshold()),
    ('classifier', LogisticRegression(solver='liblinear'))
])

score = cross_validate(pipe, X_train, y_train, cv=5)
    
print('Test set score: ' + str(np.mean(score['test_score'])))

### 9.2 Parameter tuning

#### 9.2.1 penalty

In [ ]:
pipe = Pipeline([
    ('pca', PCA(n_components=10)),
    ('selector', VarianceThreshold()),
    ('classifier', LogisticRegression(solver='liblinear', penalty='l1'))
])

score = cross_validate(pipe, X_train, y_train, cv=5)
    
print('penalty=l1 | Test set score: ' + str(np.mean(score['test_score'])))

It produces a slightly better results, so we are going to stick with l1.

### 9.3 Optimal model

In [12]:
pipe = Pipeline([
    ('pca', PCA(n_components=10)),
    ('selector', VarianceThreshold()),
    ('classifier', LogisticRegression(solver='liblinear', penalty='l1'))
])

score = cross_validate(pipe, X_train, y_train, cv=5)
    
print('Test set score: ' + str(np.mean(score['test_score'])))

## 10. Test Set Evaluation

Here we choose a model with the best parameters set from eash machine learning algorithms based on the validation set performance. We retrain the models again with full modelling data and evaluate them with the test set.

### 10.1 Logistic Regression

In [16]:
pipe = Pipeline([
    ('pca', PCA(n_components=10)),
    ('selector', VarianceThreshold()),
    ('classifier', LogisticRegression(solver='liblinear', penalty='l1'))
])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
    
print('Accuracy | Test set score: ' + str(metrics.accuracy_score(y_pred, y_test)))
print('F1 score | Test set score: ' + str(metrics.f1_score(y_pred, y_test, average='macro')))

### 10.2 K nearest neighbors

In [17]:
pipe = Pipeline([
    ('pca', PCA(n_components=10)),
    ('selector', VarianceThreshold()),
    ('classifier', KNeighborsClassifier(n_neighbors=19, metric="correlation"))
])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
    
print('Accuracy | Test set score: ' + str(metrics.accuracy_score(y_pred, y_test)))
print('F1 score | Test set score: ' + str(metrics.f1_score(y_pred, y_test, average='macro')))

### 10.3 Random forest

In [19]:
pipe = Pipeline([
    ('pca', PCA(n_components=10)),
    ('selector', VarianceThreshold()),
    ('classifier', RandomForestClassifier(random_state=1, n_estimators=300))
])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
    
print('Accuracy | Test set score: ' + str(metrics.accuracy_score(y_pred, y_test)))
print('F1 score | Test set score: ' + str(metrics.f1_score(y_pred, y_test, average='macro')))

## 11. Summary

The accuracy scored and the f1 score differ quite a bit between algorithms. <br>
The K nearest neighbor perfmormed the worst. <br>

The best approach was Random forest classifier. <br>
The final parameter is n_estimators=300.